# Figure 5 Bar Chart: PFT-level Percentage Differences

Data source: 5-year averaged (2010-2014) CLM5 output from `G:/Hangkai/CTH_ET project/Final_data/`

In [ ]:
import netCDF4
import os
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Load all data from 5-year averaged Final_data
# ============================================================
file_path = r'G:\Hangkai\CTH_ET project\Final_data'

scenarios_EX = ['CLM Default', 'GEDI Max', 'GEDI Mean', 'GEDI Median']
variable_names = ['FCEV', 'FCTR', 'FGEV']

scenario_mapping_EX = {
    'CLM Default': 'CLM Default.nc',
    'GEDI Max': 'GEDI Max.nc',
    'GEDI Mean': 'GEDI Mean.nc',
    'GEDI Median': 'GEDI Median.nc',
}

# Load PFT structure from Final_data
with netCDF4.Dataset(os.path.join(file_path, 'CLM Default.nc'), 'r') as ds:
    pfts1d_itype_veg = np.array(ds.variables['pfts1d_itype_veg'][:])
    pfts1d_ixy       = np.array(ds.variables['pfts1d_ixy'][:])
    pfts1d_jxy       = np.array(ds.variables['pfts1d_jxy'][:])
    pfts1d_wtgcell   = np.array(ds.variables['pfts1d_wtgcell'][:])
    area             = np.array(ds.variables['area'][:])

# Handle potential time dimension
if pfts1d_itype_veg.ndim > 1:
    pfts1d_itype_veg = pfts1d_itype_veg[0, :]
    pfts1d_ixy = pfts1d_ixy[0, :]
    pfts1d_jxy = pfts1d_jxy[0, :]
    pfts1d_wtgcell = pfts1d_wtgcell[0, :]

# Load lat/lon for grid shape
file_path_input = r'H:\CLM_input'
with netCDF4.Dataset(os.path.join(file_path_input, 'mean.nc'), 'r') as ds:
    lat2d = ds.variables['LATIXY'][:]
    lon2d = ds.variables['LONGXY'][:]
lon2d = lon2d - 180

pft_names = {
    "NET Temp": 1, "NET Bor": 2, "NDT Bor": 3,
    "BET Trop": 4, "BET Temp": 5,
    "BDT Trop": 6, "BDT Temp": 7, "BDT Bor": 8,
}

# Load ET data
data_EX = {}
for scenario in scenario_mapping_EX:
    data_EX[scenario] = {}
    nc_file = os.path.join(file_path, scenario_mapping_EX[scenario])
    with netCDF4.Dataset(nc_file, 'r') as ds:
        for variable in variable_names:
            data_EX[scenario][variable] = ds.variables[variable][:]

# Calculate ET for each scenario
for scenario in data_EX:
    data_EX[scenario]['ET'] = (data_EX[scenario]['FCEV'] +
                                data_EX[scenario]['FCTR'] +
                                data_EX[scenario]['FGEV'])

print('Data loaded successfully.')

In [ ]:
# ============================================================
# Compute area-weighted percentage differences by PFT
# ============================================================
components = ['FCEV', 'FCTR', 'FGEV', 'ET']
gedi_scenarios = ['GEDI Max', 'GEDI Mean', 'GEDI Median']
baseline = 'CLM Default'

# Flatten area if needed for indexing
area_flat = area.ravel()

pft_stats = {comp: {pft: {} for pft in pft_names} for comp in components}

for comp in components:
    for pft_name, pft_id in pft_names.items():
        pft_indexes = np.where(pfts1d_itype_veg == pft_id)[0]
        if len(pft_indexes) == 0:
            continue

        base_vals = np.mean(data_EX[baseline][comp], axis=0) * pfts1d_wtgcell
        base_pft = base_vals[pft_indexes]

        # Get area weights
        area_weights = np.zeros(len(pft_indexes))
        for i, idx in enumerate(pft_indexes):
            row = int(pfts1d_jxy[idx]) - 1
            col = int(pfts1d_ixy[idx]) - 1
            flat_idx = row * len(lon2d[0]) + col
            if flat_idx < len(area_flat):
                area_weights[i] = area_flat[flat_idx]
            else:
                area_weights[i] = 1.0

        for scenario in gedi_scenarios:
            scen_vals = np.mean(data_EX[scenario][comp], axis=0) * pfts1d_wtgcell
            scen_pft = scen_vals[pft_indexes]

            valid = np.abs(base_pft) > 1e-10
            if np.sum(valid) == 0:
                pft_stats[comp][pft_name][scenario] = (0, 0, 0)
                continue

            pct_diff = (scen_pft[valid] - base_pft[valid]) / base_pft[valid] * 100
            weights = area_weights[valid]

            weighted_mean = np.average(pct_diff, weights=weights)
            q25 = np.percentile(pct_diff, 25)
            q75 = np.percentile(pct_diff, 75)

            pft_stats[comp][pft_name][scenario] = (weighted_mean, q25, q75)

print('PFT statistics computed.')

In [ ]:
# ============================================================
# Figure 5 Bar Chart: PFT-level percentage differences
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10), dpi=300)

panel_order = ['FCEV', 'FGEV', 'FCTR', 'ET']
panel_titles = ['(a) Canopy Evaporation', '(b) Ground Evaporation',
                '(c) Canopy Transpiration', '(d) Total Evapotranspiration']

colors = {'GEDI Max': '#E8A838', 'GEDI Mean': '#E07070', 'GEDI Median': '#7CAE7A'}
pft_list = list(pft_names.keys())
x = np.arange(len(pft_list))
bar_width = 0.25

for idx, (comp, title) in enumerate(zip(panel_order, panel_titles)):
    ax = axes[idx // 2, idx % 2]

    for j, scenario in enumerate(gedi_scenarios):
        means = []
        lower_err = []
        upper_err = []

        for pft in pft_list:
            if scenario in pft_stats[comp][pft]:
                m, q25, q75 = pft_stats[comp][pft][scenario]
                means.append(m)
                lower_err.append(m - q25)
                upper_err.append(q75 - m)
            else:
                means.append(0)
                lower_err.append(0)
                upper_err.append(0)

        offset = (j - 1) * bar_width
        bars = ax.bar(x + offset, means, bar_width, label=scenario,
                      color=colors[scenario], edgecolor='black', linewidth=0.5,
                      yerr=[lower_err, upper_err], capsize=2, error_kw={'linewidth': 0.8})

    ax.axhline(y=0, color='black', linewidth=0.8, linestyle='-')
    ax.set_xticks(x)
    ax.set_xticklabels(pft_list, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('Relative Difference (%)', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold', loc='left')
    ax.tick_params(axis='y', labelsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    if idx == 0:
        ax.legend(fontsize=10, loc='upper right', frameon=True, edgecolor='gray')

plt.tight_layout()
plt.savefig('figures/Figure_5_Bar_Chart.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/Figure_5_Bar_Chart.tiff', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 5 Bar Chart saved.')